# 02 — Scenario Analysis
Run the network under demand growth, freight inflation, and disruption scenarios.

In [2]:
%cd /content
!test -d supply-chain-network-optimization || git clone https://github.com/hemanthkarthikd/supply-chain-network-optimization.git
%cd /content/supply-chain-network-optimization/notebooks

/content
/content/supply-chain-network-optimization/notebooks


In [3]:
# Run 01_network_optimization.ipynb first, or copy its solve_network() function here.
# Recommended scenarios:
# 1. Baseline
# 2. Demand +15%
# 3. Transport Cost +20%
# 4. Highest-volume DC unavailable
# 5. Max 4 DCs
#
# The repository already includes data/processed/scenario_summary.csv
# generated from these scenarios for Power BI and portfolio documentation.

In [5]:
import pandas as pd
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

plants = pd.read_csv("../data/raw/plants.csv")
dcs = pd.read_csv("../data/raw/distribution_centers.csv")
customers = pd.read_csv("../data/raw/customer_zones.csv")
pdc = pd.read_csv("../data/raw/plant_to_dc_costs.csv")
dcc = pd.read_csv("../data/raw/dc_to_customer_costs.csv")


def solve_network(
    demand_multiplier=1.0,
    transport_multiplier=1.0,
    unavailable_dcs=None,
    max_open_dcs=None
):
    unavailable_dcs = set(unavailable_dcs or [])

    P = list(plants["plant_id"])
    D = list(dcs["dc_id"])
    C = list(customers["customer_id"])

    x_keys = [(p, d) for p in P for d in D]
    z_keys = [(d, c) for d in D for c in C]

    var_keys = (
        [("x",) + k for k in x_keys]
        + [("z",) + k for k in z_keys]
        + [("y", d) for d in D]
    )

    idx = {k: i for i, k in enumerate(var_keys)}
    n = len(var_keys)

    objective = np.zeros(n)

    pdc_cost = {
        (r.plant_id, r.dc_id): r.transport_cost_per_unit
        for _, r in pdc.iterrows()
    }

    dcc_cost = {
        (r.dc_id, r.customer_id): r.transport_cost_per_unit
        for _, r in dcc.iterrows()
    }

    fixed_cost = dict(zip(dcs["dc_id"], dcs["fixed_annual_cost"]))
    production_cost = dict(
        zip(plants["plant_id"], plants["production_cost_per_unit"])
    )

    for p, d in x_keys:
        objective[idx[("x", p, d)]] = (
            production_cost[p]
            + pdc_cost[(p, d)] * transport_multiplier
        )

    for d, c in z_keys:
        objective[idx[("z", d, c)]] = (
            dcc_cost[(d, c)] * transport_multiplier
        )

    for d in D:
        objective[idx[("y", d)]] = fixed_cost[d]

    A = []
    lower_constraint = []
    upper_constraint = []

    plant_capacity = dict(
        zip(plants["plant_id"], plants["capacity_units"])
    )

    dc_capacity = dict(
        zip(dcs["dc_id"], dcs["capacity_units"])
    )

    demand = dict(
        zip(
            customers["customer_id"],
            customers["demand_units"] * demand_multiplier
        )
    )

    # Plant capacity
    for p in P:
        row = np.zeros(n)

        for d in D:
            row[idx[("x", p, d)]] = 1

        A.append(row)
        lower_constraint.append(-np.inf)
        upper_constraint.append(plant_capacity[p])

    # DC flow balance
    for d in D:
        row = np.zeros(n)

        for p in P:
            row[idx[("x", p, d)]] = 1

        for c in C:
            row[idx[("z", d, c)]] -= 1

        A.append(row)
        lower_constraint.append(0)
        upper_constraint.append(0)

    # DC capacity
    for d in D:
        row = np.zeros(n)

        for c in C:
            row[idx[("z", d, c)]] = 1

        row[idx[("y", d)]] = -dc_capacity[d]

        A.append(row)
        lower_constraint.append(-np.inf)
        upper_constraint.append(0)

    # Customer demand
    for c in C:
        row = np.zeros(n)

        for d in D:
            row[idx[("z", d, c)]] = 1

        A.append(row)
        lower_constraint.append(demand[c])
        upper_constraint.append(demand[c])

    # Optional DC-count limit
    if max_open_dcs is not None:
        row = np.zeros(n)

        for d in D:
            row[idx[("y", d)]] = 1

        A.append(row)
        lower_constraint.append(-np.inf)
        upper_constraint.append(max_open_dcs)

    constraints = LinearConstraint(
        np.array(A),
        np.array(lower_constraint),
        np.array(upper_constraint)
    )

    lower_bounds = np.zeros(n)
    upper_bounds = np.full(n, np.inf)

    integrality = np.zeros(n, dtype=int)

    for d in D:
        upper_bounds[idx[("y", d)]] = (
            0 if d in unavailable_dcs else 1
        )

        integrality[idx[("y", d)]] = 1

    result = milp(
        c=objective,
        constraints=constraints,
        bounds=Bounds(lower_bounds, upper_bounds),
        integrality=integrality
    )

    if not result.success:
        return {
            "success": False,
            "total_cost": np.nan,
            "open_dcs": []
        }

    solution = result.x

    open_dcs = [
        d for d in D
        if solution[idx[("y", d)]] > 0.5
    ]

    return {
        "success": True,
        "total_cost": result.fun,
        "open_dcs": open_dcs
    }

In [6]:
baseline = solve_network()

scenarios = {
    "Baseline": baseline,
    "Demand +15%": solve_network(
        demand_multiplier=1.15
    ),
    "Transport Cost +20%": solve_network(
        transport_multiplier=1.20
    ),
    "D3 Outage": solve_network(
        unavailable_dcs=["D3"]
    ),
    "Max 4 DCs": solve_network(
        max_open_dcs=4
    )
}

results = []

for scenario_name, result in scenarios.items():
    results.append({
        "scenario": scenario_name,
        "success": result["success"],
        "total_annual_cost": round(result["total_cost"], 2),
        "open_dcs": ", ".join(result["open_dcs"]),
        "number_of_open_dcs": len(result["open_dcs"])
    })

scenario_summary = pd.DataFrame(results)

display(scenario_summary)

scenario_summary.to_csv(
    "../data/processed/scenario_summary.csv",
    index=False
)

,scenario,success,total_annual_cost,open_dcs,number_of_open_dcs
0,Baseline,True,10948295.00,"D1, D3, D4, D5",4
1,Demand +15%,True,11749570.25,"D1, D3, D4, D5",4
2,Transport Cost +20%,True,11476104.00,"D1, D3, D4, D5",4
3,D3 Outage,True,11285460.00,"D1, D2, D4, D5",4
4,Max 4 DCs,True,10948295.00,"D1, D3, D4, D5",4


In [4]:
import pandas as pd
scenario_summary = pd.read_csv("../data/processed/scenario_summary.csv")
display(scenario_summary)

,scenario,success,total_annual_cost,open_dcs,number_of_open_dcs
0,Baseline,True,10948295.00,"D1, D3, D4, D5",4
1,Demand +15%,True,11749570.25,"D1, D3, D4, D5",4
2,Transport Cost +20%,True,11476104.00,"D1, D3, D4, D5",4
3,D3 Outage,True,11285460.00,"D1, D2, D4, D5",4
4,Max 4 DCs,True,10948295.00,"D1, D3, D4, D5",4
